# **LaTeX table with the BAO-fit results**

This notebook shows how to create a LaTeX table with the BAO-fit results from a set of mocks. It also makes some plots

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
import scipy
from utils_data import GetThetaLimits
from utils_baofit import BAOFitInitializer
from utils_inference import BAOFitChecker
%matplotlib inline

# helper to select only needed keys for a function
def subset(cfg, *keys):
    return {k: cfg[k] for k in keys}

recon_list = ["pre", "post"]
pow_broadband = [-1, 0]
# pow_broadband = [0]
tracer_list = ["LRG", "ELG2", "QSO"]
# tracer_list = ["ELG2"]
# tracer_list = ["QSO"]

n_mocks = 25
mock_id_list = ["mean"] + list(range(n_mocks))

alpha_type = "alpha_wigg_only"
alpha_type = "alpha_wigg_nowigg"

def get_bins_removed_list(tracer, delta_z):
    if delta_z == 0.02:
        if tracer == "LRG":
            nbins = 35
            configs = [
                ("LRG-full", []),
                ("LRG1", list(range(10, 35))),
                ("LRG2", list(range(0, 10)) + list(range(20, 35))),
                ("LRG3", list(range(0, 20))),
            ]
        elif tracer == "LRG+ELG":
            nbins = 15
            configs = [
                (tracer, []),
            ]
        elif tracer == "ELG":
            nbins = 40
            configs = [
                ("ELG-full", []),
                ("ELG1", list(range(15, 40))),
                ("ELG2", list(range(0, 15))),
            ]
    elif delta_z == 0.05:
        if tracer == "ELG2":
            nbins = 10
            configs = [
                (tracer, []),
            ]
        elif tracer == "QSO":
            nbins = 26
            configs = [
                (tracer, []),
            ]
    else:
        return []
    # # Individual redshift bins
    # for i in range(nbins):
    #     excluded = [j for j in range(nbins) if j != i]
    #     configs.append((f"{tracer} bin {i}", excluded))
    return configs

fit_results = {}
all_params_results = {}
mock_id_detected_list = {}
mock_id_nondetected_list = {}
significance_detection = {}

for recon, tracer in product(recon_list, tracer_list):
    if tracer in ["ELG2", "QSO"]:
        delta_z = 0.05
    else:
        delta_z = 0.02
    
    dataset = f"DESIY1_{tracer}_Abacus_altmtl_{recon}_deltaz{delta_z}"
    bins_removed_list = get_bins_removed_list(tracer, delta_z)

    if tracer in ["ELG", "ELG2", "QSO"]:
        theta_width = 3
    else:
        theta_width = 6

    theta_min, theta_max = GetThetaLimits(dataset=dataset, nz_flag="mocks", dynamical_theta_limits=True, theta_width=theta_width, code_path=code_path).get_theta_limits()
    
    for tracer_label, bins_removed in bins_removed_list:
        print(tracer_label, recon)

        fit_results[tracer_label, recon] = {}
        all_params_results[tracer_label, recon] = {}
        mock_id_detected_list[tracer_label, recon] = []
        mock_id_nondetected_list[tracer_label, recon] = []
        significance_detection[tracer_label, recon] = {}

        for mock_id in mock_id_list:
            # 1. Arguments
            base_config = {
                "dataset": dataset,
                "weight_type": 1, # it will not be used...
                "mock_id": mock_id,
                "nz_flag": "mocks",
                "cov_type": "mocks",
                "cosmology_template": "desifid",
                "cosmology_covariance": "desifid",
                "delta_theta": 0.2,
                "theta_min": theta_min,
                "theta_max": theta_max,
                "pow_broadband": pow_broadband,
                "bins_removed": bins_removed,
                "alpha_min": 0.8,
                "alpha_max": 1.2,
                "alpha_type": alpha_type,
                "code_path": code_path,
                "save_path": save_path,
            }
            
            baofit_checker = BAOFitChecker(**base_config) # this will print information related to the detection
    
            if baofit_checker.is_detection:
    
                config = {
                    **base_config,
                    "include_wiggles": "",
                }
    
                # 2. BAO fit initializer. This basically creates the path to load the results
                baofit_initializer = BAOFitInitializer(
                    **subset(config, "include_wiggles", "dataset", "weight_type", "mock_id",
                             "nz_flag", "cov_type", "cosmology_template", "cosmology_covariance",
                             "delta_theta", "theta_min", "theta_max", "pow_broadband",
                             "bins_removed", "alpha_min", "alpha_max", "alpha_type", "save_path"),
                    verbose=False,
                )
    
                fit_results[tracer_label, recon][mock_id] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "fit_results.txt"))
                all_params_results[tracer_label, recon][mock_id] = np.load(os.path.join(baofit_initializer.path_baofit, "all_params_bestfit.npy"), allow_pickle=True).item()
                if mock_id != "mean":
                    mock_id_detected_list[tracer_label, recon].append(mock_id)
                significance_detection[tracer_label, recon][mock_id] = baofit_checker.significance
                # print(f"alpha = {fit_results[tracer_label, recon][mock_id][0]} +- {fit_results[tracer_label, recon][mock_id][1]}")
            else:
                mock_id_nondetected_list[tracer_label, recon].append(mock_id)
                
        print()


In [ ]:
summary_data = []

for (tracer, recon), values in fit_results.items():
    alpha_mocks = []
    sigma_alpha_mocks = []
    chi2s = []
    dofs = []

    for mock_id, results in fit_results[tracer, recon].items():
        alpha, sigma_alpha, chi2, dof = results
        if mock_id == "mean":
            alpha_mean_mocks = alpha
            sigma_alpha_mean_mocks = sigma_alpha
            chi2_mean_mocks = chi2
            dof_mean_mocks = dof
        else:
            alpha_mocks.append(alpha)
            sigma_alpha_mocks.append(sigma_alpha)
            chi2s.append(chi2)
            dofs.append(dof)

    alpha_mocks = np.array(alpha_mocks)
    sigma_alpha_mocks = np.array(sigma_alpha_mocks)
    chi2s = np.array(chi2s)
    dofs = np.array(dofs)

    alpha_mean = alpha_mocks.mean()
    sigma_std = alpha_mocks.std()

    upper, lower = np.percentile(alpha_mocks, [84.075, 15.825], method="linear")
    sigma_68 = (upper - lower) / 2

    mean_sigma_alpha = sigma_alpha_mocks.mean()

    frac_enc = np.mean(
        (alpha_mocks > alpha_mean - mean_sigma_alpha) &
        (alpha_mocks < alpha_mean + mean_sigma_alpha)
    ) * 100

    d_norm = (alpha_mocks - alpha_mean) / sigma_alpha_mocks
    dnorm_mean = d_norm.mean()
    dnorm_std = d_norm.std()

    mean_chi2 = chi2s.mean()
    mean_dof = dofs.mean() # they should all be the same...
    chi2_over_dof_str = f"{mean_chi2:.1f}/{int(round(mean_dof))}"

    alpha_mean_pm_sigma = f"{alpha_mean_mocks:.4f} ± {sigma_alpha_mean_mocks:.4f}"
    chi2_over_dof_str_mean = f"{chi2_mean_mocks:.1f}/{int(round(dof_mean_mocks))}"

    summary_data.append([
        tracer,
        recon,
        alpha_mean,
        sigma_std,
        # sigma_68,
        mean_sigma_alpha,
        # frac_enc,
        # dnorm_mean,
        # dnorm_std,
        chi2_over_dof_str,
        # alpha_mean_pm_sigma,
        # chi2_over_dof_str_mean
    ])

summary_df = pd.DataFrame(summary_data, columns=[
    "tracer",
    "recon",
    r"$\langle\alpha\rangle$",
    r"$\sigma_{\rm std}$",
    # r"$\sigma_{68}$",
    r"$\langle\sigma_\alpha\rangle$",
    # r"fraction encl.$\langle\alpha\rangle$ [\%]",
    # r"$\langle d_{\rm norm}\rangle$",
    # r"$\sigma_{d_{\rm norm}}$",
    r"$\langle\chi^2\rangle/$dof",
    # "Mean of mocks",
    # r"$\chi^2_{\rm mean\ mocks}/$dof",
])

# Format columns
for col in [
    r"$\langle\alpha\rangle$",
    r"$\sigma_{\rm std}$",
    # r"$\sigma_{68}$",
    r"$\langle\sigma_\alpha\rangle$"
]:
    summary_df[col] = summary_df[col].map(lambda x: f"{float(x):.4f}")

# summary_df[r"fraction encl.$\langle\alpha\rangle$ [\%]"] = summary_df[
#     r"fraction encl.$\langle\alpha\rangle$ [\%]"].map(lambda x: f"{float(x):.1f}")

# summary_df[r"$\langle d_{\rm norm}\rangle$"] = summary_df[
#     r"$\langle d_{\rm norm}\rangle$"].map(lambda x: f"{float(x):.4f}")

# summary_df[r"$\sigma_{d_{\rm norm}}$"] = summary_df[
#     r"$\sigma_{d_{\rm norm}}$"].map(lambda x: f"{float(x):.4f}")

# Output LaTeX table
latex_table = summary_df.to_latex(index=False, escape=False, column_format="l|l|" + "c" * (len(summary_df.columns) - 2))
print(latex_table)


In [ ]:
figsize = (18, 5)

for (tracer, recon), values in fit_results.items():

    # Compute means
    mean_alpha = np.mean([fit_results[tracer, recon][mock_id][0] for mock_id in fit_results[tracer, recon].keys()])
    std_alpha = np.std([fit_results[tracer, recon][mock_id][0] for mock_id in fit_results[tracer, recon].keys()])
    mean_sigma_alpha = np.mean([fit_results[tracer, recon][mock_id][1] for mock_id in fit_results[tracer, recon].keys()])
    mean_chi2 = np.mean([fit_results[tracer, recon][mock_id][2] for mock_id in fit_results[tracer, recon].keys()])
    
    # Create 1x3 subplot grid
    fig, axs = plt.subplots(1, 3, figsize=figsize)

    fig.suptitle(fr"\texttt{{{tracer, recon}}}", fontsize=20)
    
    # Histogram for alpha
    axs[0].hist([fit_results[tracer, recon][mock_id][0] for mock_id in fit_results[tracer, recon].keys()], 
                color='royalblue', edgecolor='black', alpha=0.7)
    axs[0].axvline(mean_alpha, color='black', linestyle="--", linewidth=2, label=fr'$\langle\alpha\rangle$ = {mean_alpha:.4f}')
    axs[0].set_xlabel(r'$\alpha$', fontsize=18)
    axs[0].tick_params(axis='both', labelsize=14)
    axs[0].legend(fontsize=14, loc='upper right')
    # axs[0].text(0.95, 0.85, fr'$\sigma_{{\rm std}}$ = {std_alpha:.4f}', ha='right', va='top', transform=axs[0].transAxes, fontsize=14)
    
    # Histogram for sigma_alpha
    axs[1].hist([fit_results[tracer, recon][mock_id][1] for mock_id in fit_results[tracer, recon].keys()], 
                color='darkorange', edgecolor='black', alpha=0.7)
    axs[1].axvline(mean_sigma_alpha, color='black', linestyle="--", linewidth=2, label=fr'$\langle\sigma_\alpha\rangle$ = {mean_sigma_alpha:.4f}')
    axs[1].set_xlabel(r'$\sigma_{\alpha}$', fontsize=18)
    axs[1].tick_params(axis='both', labelsize=14)
    axs[1].legend(fontsize=14)
    
    # Histogram for chi^2
    axs[2].hist([fit_results[tracer, recon][mock_id][2] for mock_id in fit_results[tracer, recon].keys()], density=True, 
                color='forestgreen', edgecolor='black', alpha=0.7)
    axs[2].axvline(mean_chi2, color='black', linestyle="--", linewidth=2, label=fr'$\langle\chi^2\rangle$ = {mean_chi2:.1f}')
    axs[2].set_xlabel(r'$\chi^2$', fontsize=18)
    axs[2].tick_params(axis='both', labelsize=14)
    
    # Overlay chi-squared distribution on the chi2 histogram
    dof = fit_results[tracer, recon]["mean"][3]
    x = np.linspace(0, dof * 2, 500)
    pdf = scipy.stats.chi2.pdf(x, dof)
    axs[2].plot(x, pdf, label=fr'$\chi^2$ dist. (dof = {int(dof)})', color='darkgreen')
    axs[2].legend(loc='upper right', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(f"plots/analysis_{tracer}_{recon}.png")
    

In [ ]:
# # Identify rows
# full_row = summary_df.iloc[0]  # Full LRG
# sub_rows = summary_df.iloc[1:4]  # LRG1, LRG2, LRG3

# # Extract arrays
# alphas = sub_rows[r"$\langle\alpha\rangle$"].astype(float).values
# sigmas = sub_rows[r"$\langle\sigma_\alpha\rangle$"].astype(float).values

# # Compute weights (inverse variance)
# weights = 1.0 / (sigmas**2)

# # Combined alpha and sigma
# alpha_comb = np.sum(alphas * weights) / np.sum(weights)
# sigma_comb = np.sqrt(1.0 / np.sum(weights))

# # Extract full LRG values
# full_alpha = float(full_row[r"$\langle\alpha\rangle$"])
# full_sigma = float(full_row[r"$\langle\sigma_\alpha\rangle$"])

# # Print comparison
# print("=== Comparison ===")
# print(f"Full LRG:    alpha = {full_alpha:.4f}, sigma_alpha = {full_sigma:.4f}")
# print(f"LRG1+LRG2+LRG3:    alpha = {alpha_comb:.4f}, sigma_alpha = {sigma_comb:.4f}")


## Let's add the data!

In [ ]:
fit_results_data = {}
significance_detection_data = {}

for recon, tracer in product(recon_list, tracer_list):
    if tracer in ["ELG2", "QSO"]:
        delta_z = 0.05
    else:
        delta_z = 0.02
    
    dataset = f"DESIY1_{tracer}_{recon}_deltaz{delta_z}"
    bins_removed_list = get_bins_removed_list(tracer, delta_z)

    if tracer in ["ELG", "ELG2", "QSO"]:
        theta_width = 3
    else:
        theta_width = 6

    theta_min, theta_max = GetThetaLimits(dataset=dataset, nz_flag="mocks", dynamical_theta_limits=True, theta_width=theta_width, code_path=code_path).get_theta_limits()
    
    for tracer_label, bins_removed in bins_removed_list:
        print(tracer_label, recon)

        # 1. Arguments
        base_config = {
            "dataset": dataset,
            "weight_type": 1, # it will not be used...
            "mock_id": "mean", # it will not be used...
            "nz_flag": "mocks",
            "cov_type": "mocks",
            "cosmology_template": "desifid",
            "cosmology_covariance": "desifid",
            "delta_theta": 0.2,
            "theta_min": theta_min,
            "theta_max": theta_max,
            "pow_broadband": pow_broadband,
            "bins_removed": bins_removed,
            "alpha_min": 0.8,
            "alpha_max": 1.2,
            "alpha_type": alpha_type,
            "code_path": code_path,
            "save_path": save_path,
        }

        baofit_checker = BAOFitChecker(**base_config) # this will print information related to the detection

        if baofit_checker.is_detection:

            config = {
                **base_config,
                "include_wiggles": "",
            }

            # 2. BAO fit initializer. This basically creates the path to load the results
            baofit_initializer = BAOFitInitializer(
                **subset(config, "include_wiggles", "dataset", "weight_type", "mock_id",
                         "nz_flag", "cov_type", "cosmology_template", "cosmology_covariance",
                         "delta_theta", "theta_min", "theta_max", "pow_broadband",
                         "bins_removed", "alpha_min", "alpha_max", "alpha_type", "save_path"),
                verbose=False,
            )

            fit_results_data[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "fit_results.txt"))
            significance_detection_data[tracer_label, recon] = baofit_checker.significance
            

In [ ]:
from matplotlib import gridspec

# for tracer in ["LRG-full", "LRG1", "LRG2", "LRG3"]:
# for tracer in ["ELG2"]:
for tracer in tracer_list:
    if tracer in ["ELG2", "QSO"]:
        delta_z = 0.05
    else:
        delta_z = 0.02
    
    bins_removed_list = get_bins_removed_list(tracer, delta_z)

    for tracer_label, bins_removed in bins_removed_list:
        print(tracer_label)

        mock_id_common = list(set(mock_id_detected_list[tracer_label, "pre"]) & set(mock_id_detected_list[tracer_label, "post"]))
        
        # --- Extract individual mock data ---
        alpha_pre = np.array([fit_results[tracer_label, "pre"][mock_id][0] for mock_id in mock_id_common])
        alpha_post = np.array([fit_results[tracer_label, "post"][mock_id][0] for mock_id in mock_id_common])
        sigma_alpha_pre = np.array([fit_results[tracer_label, "pre"][mock_id][1] for mock_id in mock_id_common])
        sigma_alpha_post = np.array([fit_results[tracer_label, "post"][mock_id][1]for mock_id in mock_id_common])
        Nsigma_pre = np.array([significance_detection[tracer_label, "pre"][mock_id] for mock_id in mock_id_common])
        Nsigma_post = np.array([significance_detection[tracer_label, "post"][mock_id]for mock_id in mock_id_common])
        
        # --- Extract "mean-of-mocks" fit ---
        alpha_pre_mean_mock = fit_results[tracer_label, "pre"]["mean"][0]
        alpha_post_mean_mock = fit_results[tracer_label, "post"]["mean"][0]
        sigma_pre_mean_mock = fit_results[tracer_label, "pre"]["mean"][1]
        sigma_post_mean_mock = fit_results[tracer_label, "post"]["mean"][1]
        
        # --- Means of individual mocks (for reference) ---
        alpha_pre_mean = alpha_pre.mean()
        alpha_post_mean = alpha_post.mean()
        sigma_pre_mean = sigma_alpha_pre.mean()
        sigma_post_mean = sigma_alpha_post.mean()
        
        # --- Extract results on data ---
        alpha_pre_data = fit_results_data[tracer_label, "pre"][0]
        alpha_post_data = fit_results_data[tracer_label, "post"][0]
        sigma_alpha_pre_data = fit_results_data[tracer_label, "pre"][1]
        sigma_alpha_post_data = fit_results_data[tracer_label, "post"][1]
        Nsigma_pre_data = significance_detection_data[tracer_label, "pre"]
        Nsigma_post_data = significance_detection_data[tracer_label, "post"]
        
        # =========================
        # Δα vs σpost/σpre
        # =========================
        delta_alpha = alpha_post - alpha_pre
        sigma_ratio = sigma_alpha_post / sigma_alpha_pre
        
        # Mean of individual mocks
        delta_alpha_mean = delta_alpha.mean()
        sigma_ratio_mean = sigma_ratio.mean()
        
        # Fit of mean-of-mocks
        delta_alpha_mean_mock = alpha_post_mean_mock - alpha_pre_mean_mock
        sigma_ratio_mean_mock = sigma_post_mean_mock / sigma_pre_mean_mock
        
        fig, ax = plt.subplots(figsize=(5,5))
        
        ax.scatter(delta_alpha, sigma_ratio, s=25, alpha=0.8, label="individual mocks")
        
        # Mean of mocks
        ax.scatter(delta_alpha_mean, sigma_ratio_mean, s=140, marker='X', color='C1', zorder=5, label="mean of mocks")
        
        # Fit of mean
        ax.scatter(delta_alpha_mean_mock, sigma_ratio_mean_mock, s=140, marker='*', color='red', zorder=6, label="fit of mean")
        
        # Results on data
        ax.scatter(alpha_post_data - alpha_pre_data, sigma_alpha_post_data / sigma_alpha_pre_data, s=140, marker='d', color='violet', zorder=6, label="data")
        
        # Reference lines
        ax.axvline(0, linestyle='--', lw=1, color='k')
        ax.axhline(1, linestyle='--', lw=1, color='k')
        
        ax.set_xlabel(r'$\alpha_{\rm post}-\alpha_{\rm pre}$', fontsize=18)
        ax.set_ylabel(r'$\sigma_{\alpha,\rm post}/\sigma_{\alpha,\rm pre}$', fontsize=18)
        
        ax.legend(loc="best", fontsize=14)
        ax.tick_params(axis='both', labelsize=14)
        ax.set_title(tracer_label, fontsize=14)
        
        plt.tight_layout()

        fig, ax = plt.subplots(figsize=(5,5))

        ax.scatter(Nsigma_pre, Nsigma_post, s=25, alpha=0.8, label="individual mocks")

        ax.scatter(Nsigma_pre_data, Nsigma_post_data, s=140, marker='d', color='violet', zorder=6, label="data")

        ax.axline((0, 0), slope=1, color='k', linestyle='--')  # x = y

        ax.set_xlabel(r'$N_\sigma$ (pre)', fontsize=18)
        ax.set_ylabel(r'$N_\sigma$ (post)', fontsize=18)

        ax.legend(loc="best", fontsize=14)
        ax.tick_params(axis='both', labelsize=14)
        ax.set_title(tracer_label, fontsize=14)
